# Origin Classification with TF-IDF

Baseline pipeline for country-level origin classification using tasting text.

In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


In [2]:
DATA_PATH = 'Data/final_coffee_reviews.csv'
RANDOM_STATE = 42
MIN_SAMPLES_PER_CLASS = 30

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head(2)

Shape: (7585, 25)


,URL,all_text,Rating,Roaster,Coffee Name,Roaster Location,Coffee Origin,Roast Level,Agtron,Est. Price,...,Flavor,Aftertaste,With Milk,Blind Assessment,Notes,Who Should Drink It,Bottom Line,Combined_Acidity,Country_all,Country
0,https://www.coffeereview.com/review/100-arabic...,87\nCaribeans\n100% Arabica Coffee from Puerto...,87.0,Caribeans,100% Arabica Coffee from Puerto Rico,"San Juan, Puerto Rico","Utuado, central Puerto Rico",Medium-Light,54/69,$17.00/8 ounces,...,8.0,7.0,NaN,bittersweet but balanced; chocolaty. dark choc...,Produced on a single farm in the central mount...,NaN,Satisfying chocolate and nut notes nearly carr...,7.0,Puerto Rico; China,Puerto Rico
1,https://www.coffeereview.com/review/100-arabic...,88\nWaka Coffee\n100% Arabica Freeze-Dried Col...,88.0,Waka Coffee,100% Arabica Freeze-Dried Colombian (Instant C...,"Los Angeles, California",Colombia,NaN,0/0,$10.99/8 single-serve packets,...,8.0,8.0,NaN,evaluated at proportions of 5 grams of instant...,The green coffee for this product was produced...,NaN,A appealing 100% Colombia coffee in instant fo...,7.0,Colombia,Colombia


In [3]:
df["origin_country"] = (
    df["Coffee Origin"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.split()
    .str[-1]          # take last word
    .replace("", pd.NA)
)

In [4]:
df["origin_country"] = df["origin_country"].str.title()


In [5]:
import re
import pandas as pd

def extract_country(origin):
    if pd.isna(origin):
        return None

    s = str(origin).strip().strip(".")
    if s == "":
        return None

    # remove unusable labels
    if re.search(r"not disclosed|blend|various|multiple", s, flags=re.I):
        return None

    # split common separators, take last part (usually country)
    parts = [p.strip() for p in re.split(r"[,;/]", s) if p.strip()]
    if not parts:
        return None

    country = parts[-1]

    # optional normalization
    mapping = {
        "usa": "United States",
        "u.s.a": "United States",
        "u.s.a.": "United States",
        "hawaii": "United States",
    }
    return mapping.get(country.lower(), country.title())

df["origin_country"] = df["Coffee Origin"].apply(extract_country)


In [6]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def extract_country(origin: str) -> str | None:
    if not isinstance(origin, str):
        return None
    t = origin.strip().strip('.')
    if t == '':
        return None
    if re.search(r'not disclosed|blend|various', t, flags=re.IGNORECASE):
        return None

    parts = [p.strip() for p in re.split(r'[,;/]', t) if p.strip()]
    if not parts:
        return None
    country = parts[-1]

    country_map = {
        'usa': 'United States',
        'u.s.a.': 'United States',
        'united states': 'United States',
        'hawaii': 'United States',
    }

    key = country.lower()
    return country_map.get(key, country.title())

In [7]:
# Build text input (tasting-focused)
df['text_main'] = df['Blind Assessment'].fillna('').map(normalize_text)

# Build country-level target
df['origin_country'] = df['Country'].map(extract_country)

# Keep valid rows only
work = df[(df['text_main'].str.len() >= 30) & (df['origin_country'].notna())].copy()

# Remove tiny classes for a stable baseline
counts = work['origin_country'].value_counts()
valid_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index
work = work[work['origin_country'].isin(valid_classes)].copy()

print('Rows after filtering:', len(work))
print('Num classes:', work['origin_country'].nunique())
work['origin_country'].value_counts().head(15)

Rows after filtering: 7437
Num classes: 25


origin_country
Ethiopia         1996
Colombia          988
Kenya             699
Guatemala         561
Indonesia         452
Costa Rica        373
Panama            354
United States     302
El Salvador       240
Brazil            200
Rwanda            176
Peru              130
Nicaragua         125
Honduras          123
Mexico            101
Name: count, dtype: int64

In [8]:
X = work['text_main']
y = work['origin_country']

# 70/15/15 split (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', len(X_train), 'Val:', len(X_val), 'Test:', len(X_test))

Train: 5205 Val: 1116 Test: 1116


In [9]:
tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.90,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=3000,
        solver='saga',
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

tfidf_lr.fit(X_train, y_train)

g:\Anaconda\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=3000,
                                    n_jobs=-1, random_state=42,
                                    solver='saga'))])

In [10]:
def evaluate(model, X_data, y_true, split_name='split'):
    y_pred = model.predict(X_data)
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    print(f'== {split_name} ==')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-macro:  {f1:.4f}')

    return y_pred


_ = evaluate(tfidf_lr, X_val, y_val, 'Validation')
y_test_pred = evaluate(tfidf_lr, X_test, y_test, 'Test')

print('\nTop classes report (test):')
top_labels = y_test.value_counts().head(15).index
print(classification_report(y_test, y_test_pred, labels=top_labels, zero_division=0))

== Validation ==
Accuracy:  0.1828
Precision: 0.1130
Recall:    0.1402
F1-macro:  0.1055
== Test ==
Accuracy:  0.1971
Precision: 0.1299
Recall:    0.1528
F1-macro:  0.1209

Top classes report (test):
               precision    recall  f1-score   support

     Ethiopia       0.55      0.17      0.26       300
     Colombia       0.34      0.11      0.16       148
        Kenya       0.57      0.46      0.51       105
    Guatemala       0.15      0.06      0.09        84
    Indonesia       0.30      0.57      0.40        68
   Costa Rica       0.14      0.09      0.11        56
       Panama       0.14      0.30      0.19        53
United States       0.10      0.11      0.10        45
  El Salvador       0.11      0.14      0.12        36
       Brazil       0.16      0.40      0.23        30
       Rwanda       0.07      0.11      0.08        27
         Peru       0.08      0.15      0.10        20
     Honduras       0.00      0.00      0.00        19
    Nicaragua       0.03     

In [11]:
# Optional: preview predictions
preview = pd.DataFrame({
    'text': X_test.head(5).values,
    'true_origin': y_test.head(5).values,
    'pred_origin': tfidf_lr.predict(X_test.head(5))
})
preview

,text,true_origin,pred_origin
0,richly berry and citrus-toned; bright and live...,Taiwan,Yemen
1,"sweet-toned, fruit-forward. raspberry, cedar, ...",United States,Honduras
2,"evaluated as espresso. richly chocolaty, nut-t...",Colombia,Ethiopia
3,"balanced, crisply fruity. black cherry, pink g...",Costa Rica,Yemen
4,"richly sweet, subtly floral. butterscotch, hop...",Rwanda,Burundi


In [12]:
tfidf_svm = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.90,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(
        C=1.0,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

tfidf_svm.fit(X_train, y_train)


g:\Anaconda\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf', LinearSVC(class_weight='balanced', random_state=42))])

In [13]:
_ = evaluate(tfidf_svm, X_val, y_val, 'Validation (Linear SVM)')
y_test_pred_svm = evaluate(tfidf_svm, X_test, y_test, 'Test (Linear SVM)')

print('\nTop classes report (test, Linear SVM):')
top_labels = y_test.value_counts().head(15).index
print(classification_report(y_test, y_test_pred_svm, labels=top_labels, zero_division=0))


== Validation (Linear SVM) ==
Accuracy:  0.2545
Precision: 0.1222
Recall:    0.1380
F1-macro:  0.1261
== Test (Linear SVM) ==
Accuracy:  0.2661
Precision: 0.1414
Recall:    0.1474
F1-macro:  0.1405

Top classes report (test, Linear SVM):
               precision    recall  f1-score   support

     Ethiopia       0.49      0.36      0.42       300
     Colombia       0.31      0.19      0.23       148
        Kenya       0.41      0.51      0.46       105
    Guatemala       0.26      0.21      0.23        84
    Indonesia       0.34      0.57      0.43        68
   Costa Rica       0.16      0.12      0.14        56
       Panama       0.14      0.19      0.16        53
United States       0.11      0.16      0.13        45
  El Salvador       0.05      0.06      0.05        36
       Brazil       0.21      0.27      0.24        30
       Rwanda       0.09      0.11      0.10        27
         Peru       0.15      0.20      0.17        20
     Honduras       0.04      0.05      0.04  